In [2]:
import pandas as pd
import polars as pl
from datetime import date
from rake_nltk import Rake

# final raw pubmed data (no abstract or author)

In [27]:
pm_df = pd.read_csv("/Users/shantibrodnick/Downloads/510Capstone/GenderDisparityinResearch/data/raw/final_raw_pubmed_data.csv")

In [ ]:
pm_df.info()
# 10,206,788 rows

<class 'pandas.DataFrame'>
RangeIndex: 10206788 entries, 0 to 10206787
Data columns (total 5 columns):
 #   Column      Dtype
---  ------      -----
 0   Unnamed: 0  int64
 1   title       str  
 2   journal     str  
 3   date        str  
 4   doi         str  
dtypes: int64(1), str(4)
memory usage: 2.0 GB


In [4]:
pm_df

,Unnamed: 0,title,journal,date,doi
0,0,Anti-tumor necrosis factor-alpha antibody trea...,Rheumatology international,2006-10-20,10.1007/s00296-006-0241-1
1,1,[Quantification of expression of leukotriene B...,Beijing da xue xue bao. Yi xue ban = Journal o...,2006-10-28,NaN
2,2,Interleukin-7 induced immunopathology in arthr...,Annals of the rheumatic diseases,2006-10-14,10.1136/ard.2006.058479
3,3,High-throughput quantitation of metabolically ...,"Methods in molecular biology (Clifton, N.J.)",2006-10-31,10.1385/1-59745-167-3:267
4,4,Pulsed electrical stimulation to defer TKA in ...,Orthopedics,2006-10-26,10.3928/01477447-20061001-13
...,...,...,...,...,...
10206783,10206783,Reframing precision nutrition in irritable bow...,Frontiers in immunology,2026-06-15,10.3389/fimmu.2026.1809221
10206784,10206784,Serum neurofilament light chain in paediatric ...,"Multiple sclerosis (Houndmills, Basingstoke, E...",2026-06-23,10.1177/13524585261450816
10206785,10206785,Solvent-free engineering of a co-amorphous efa...,International journal of pharmaceutics,2026-06-11,10.1016/j.ijpharm.2026.127068
10206786,10206786,Longitudinal assessment of myocardial involvem...,Frontiers in cardiovascular medicine,2026-06-03,10.3389/fcvm.2026.1725291


In [8]:
pm_df = pm_df.drop(['Unnamed: 0'], axis = 1)

In [9]:
pm_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10206788 entries, 0 to 10206787
Data columns (total 4 columns):
 #   Column   Dtype
---  ------   -----
 0   title    str  
 1   journal  str  
 2   date     str  
 3   doi      str  
dtypes: str(4)
memory usage: 2.0 GB


In [12]:
pm_df['title'].head().tolist()

['Anti-tumor necrosis factor-alpha antibody treatment reduces serum CXCL16 levels in patients with rheumatoid arthritis.',
 '[Quantification of expression of leukotriene B4 inducing tumor necrosis factor-alpha and interleukin-1beta at mRNA level in synovial membrane cells of rheumatoid arthritis by real-time quantitative PCR].',
 'Interleukin-7 induced immunopathology in arthritis.',
 'High-throughput quantitation of metabolically labeled anionic glycoconjugates by scintillation proximity assay utilizing binding to cationic dyes.',
 'Pulsed electrical stimulation to defer TKA in patients with knee osteoarthritis.']

# cleaned_raw_pubmed_data.csv into LAZYFRAME

In [3]:
raw_df = pl.scan_csv("/Users/shantibrodnick/Downloads/510Capstone/GenderDisparityinResearch/notebooks/Article_Data_APIs/data/cleaned_raw_pubmed_data.csv")

In [5]:
raw_df.select(pl.len()).collect().item()
# 10,206,788 rows

10206788

In [28]:
raw_df.head(2).collect()

title,abstract,journal,date,authors,doi
str,str,str,str,str,str
"""Anti-tumor necrosis factor-alp…","""The aim of this study was to a…","""Rheumatology international""","""2006-10-20""","""['YasunoriKageyama', 'EijiTori…","""10.1007/s00296-006-0241-1"""
"""[Quantification of expression …","""To investigate quantification …","""Beijing da xue xue bao. Yi xue…","""2006-10-28""","""['Zhan-kunChen', 'Hou-shanLv']""",null


## Date Cleaning for that one column

In [33]:
# Find all the length of all unique dates
raw_df.select(pl.col('date').cast(pl.String).str.len_chars()).unique().collect()
# 4, 10, and 98
# 98 is odd

date
u32
10
4
98


In [34]:
# The 98 was the formatting issue which is taken care of in the pubmed_final_data_export.ipynb
raw_df.filter(pl.col('date').cast(pl.String).str.len_chars() == 98).collect()

title,abstract,journal,date,authors,doi
str,str,str,str,str,str
"""Comparing Essure® and Tubal Li…","""Hundreds of thousands of US wo…","""2021""","""['Aileen M.Gariepy', 'Daniel J…","""10.25302/10.2021.CER.160936359""",null


In [35]:
# Define row that is all shifted over
row = pl.col('date').cast(pl.String).str.len_chars() == 98

In [36]:
# Replace like we did in pubmed_final_data_export
f_raw_df = raw_df.with_columns([
    pl.when(row).then(pl.col('date')).otherwise(pl.col('authors')).alias('authors'),
    pl.when(row).then(pl.col('journal')).otherwise(pl.col('date')).alias('date'),
    pl.when(row).then(None).otherwise(pl.col('journal')).alias('journal')
])

In [37]:
# Check to see if the row changed
f_raw_df.filter(pl.col('doi') == "10.25302/10.2021.CER.160936359").collect()
# YAY IT DID

title,abstract,journal,date,authors,doi
str,str,str,str,str,str
"""Comparing Essure® and Tubal Li…","""Hundreds of thousands of US wo…",null,"""2021""","""['Aileen M.Gariepy', 'Daniel J…","""10.25302/10.2021.CER.160936359"""


In [38]:
# Find the row with 4
raw_df.filter(pl.col('date').cast(pl.String).str.len_chars() == 4).collect()

title,abstract,journal,date,authors,doi
str,str,str,str,str,str
"""Molecular Imaging and Contrast…","""The Molecular Imaging and Cont…",null,"""2004""","""[]""",null
"""Screening for Hereditary Hemoc…","""To assess evidence sufficiency…",null,"""2006""","""['Evelyn PWhitlock', 'Betsy AG…",null
"""WHO Guideline on Syphilis Scre…","""Since the publication of the W…",null,"""2017""","""[]""",null
"""Using nationwide ‘big data’ fr…","""Electronic health records (EHR…",null,"""2017""","""['HarryHemingway', 'Gene SFede…","""10.3310/pgfar05040"""
"""Can Coping-Skills Training Hel…","""Many survivors of critical ill…",null,"""2018""","""['Christopher E.Cox', 'Catheri…","""10.25302/9.2018.CER.195"""
…,…,…,…,…,…
"""Recommendations for Drug Thera…","""Evidence-informed recommendati…",null,"""2013""","""[]""",null
"""Evidence review for previous c…","""The aim of this review is to d…",null,"""2019""","""['NoneNone']""",null
"""WHO consolidated guidelines fo…","""Respiratory illness remains a …",null,"""2026""","""[]""",null


In [39]:
# Standardize the date column to make sure it has 10
ff_raw_df = f_raw_df.with_columns(
    pl.when(pl.col('date').cast(pl.String).str.len_chars() == 4).then(pl.col('date').cast(pl.String) + "-01-01")
    .otherwise(pl.col('date').cast(pl.String))
)

In [40]:
# Check to see if it worked
ff_raw_df.select(pl.col('date').cast(pl.String).str.len_chars()).unique().collect()
# Yay it worked!

date
u32
10


In [43]:
ff_raw_df.select(pl.col('date').max()).collect()

date
str
"""2026-07-01"""


In [55]:
ff_raw_df.select(pl.col('date').min()).collect()

date
str
"""1781-06-01"""


### Change date to datetime

In [44]:
# Change rows to datetime
dt_raw_df = ff_raw_df.with_columns(
    pl.col('date').str.strptime(pl.Date, format = "%Y-%m-%d")
    )

In [45]:
#pl.Config.restore_defaults()
dt_raw_df.head(2).collect()

title,abstract,journal,date,authors,doi
str,str,str,date,str,str
"""Anti-tumor necrosis factor-alp…","""The aim of this study was to a…","""Rheumatology international""",2006-10-20,"""['YasunoriKageyama', 'EijiTori…","""10.1007/s00296-006-0241-1"""
"""[Quantification of expression …","""To investigate quantification …","""Beijing da xue xue bao. Yi xue…",2006-10-28,"""['Zhan-kunChen', 'Hou-shanLv']""",null


### Drop authors column

In [58]:
dt_drop_raw_df = dt_raw_df.drop(['authors'])

## Now I want to filter for only years between 1948 - 2026

In [26]:
# Filter between 1948 - 2026
fil_raw_df = dt_raw_df.filter(pl.col('date').is_between(date(1948, 1, 1), date(2026, 6, 30)))

In [ ]:
# Check row count now
fil_raw_df.select(pl.len()).collect().item()
# 10,185,407
# Too big

10185407

In [ ]:
#raw_df_00_26.select(pl.len()).collect().item()
# 9,380,169
# #raw_df_00_26.collect().estimated_size("mb")
# 14 GB

9380169

In [55]:
#pl.Config(fmt_str_lengths=2000, tbl_rows=2)
fil_raw_df.head(2).collect()

title,abstract,journal,date,authors,doi
str,str,str,date,str,str
"""Anti-tumor necrosis factor-alp…","""The aim of this study was to a…","""Rheumatology international""",2006-10-20,"""['YasunoriKageyama', 'EijiTori…","""10.1007/s00296-006-0241-1"""
"""[Quantification of expression …","""To investigate quantification …","""Beijing da xue xue bao. Yi xue…",2006-10-28,"""['Zhan-kunChen', 'Hou-shanLv']""",null


# Thought
## What if I instead of filtering for years, and only doing 2000 - 2026 or even less, I can try and extract keywords from the abstract of the lazy frame and then save that as a csv.

### Test

In [15]:
# Make a small test lazyframe
test = raw_df.head(5).collect()

In [16]:
test

title,abstract,journal,date,authors,doi
str,str,str,str,str,str
"""Anti-tumor necrosis factor-alp…","""The aim of this study was to a…","""Rheumatology international""","""2006-10-20""","""['YasunoriKageyama', 'EijiTori…","""10.1007/s00296-006-0241-1"""
"""[Quantification of expression …","""To investigate quantification …","""Beijing da xue xue bao. Yi xue…","""2006-10-28""","""['Zhan-kunChen', 'Hou-shanLv']""",null
"""Interleukin-7 induced immunopa…","""Interleukin (IL)-7 is a potent…","""Annals of the rheumatic diseas…","""2006-10-14""","""['S A YHartgring', 'J W JBijls…","""10.1136/ard.2006.058479"""
"""High-throughput quantitation o…","""Rapid, quantitative methods su…","""Methods in molecular biology (…","""2006-10-31""","""['Karen JRees-Milton', 'Tassos…","""10.1385/1-59745-167-3:267"""
"""Pulsed electrical stimulation …",null,"""Orthopedics""","""2006-10-26""","""['Michael AMont', 'David SHung…","""10.3928/01477447-20061001-13"""


In [17]:
# View lenngth of each abstract
test.with_columns(
    pl.col('abstract').str.len_chars()
)

title,abstract,journal,date,authors,doi
str,u32,str,str,str,str
"""Anti-tumor necrosis factor-alp…",925,"""Rheumatology international""","""2006-10-20""","""['YasunoriKageyama', 'EijiTori…","""10.1007/s00296-006-0241-1"""
"""[Quantification of expression …",1589,"""Beijing da xue xue bao. Yi xue…","""2006-10-28""","""['Zhan-kunChen', 'Hou-shanLv']""",null
"""Interleukin-7 induced immunopa…",1425,"""Annals of the rheumatic diseas…","""2006-10-14""","""['S A YHartgring', 'J W JBijls…","""10.1136/ard.2006.058479"""
"""High-throughput quantitation o…",1730,"""Methods in molecular biology (…","""2006-10-31""","""['Karen JRees-Milton', 'Tassos…","""10.1385/1-59745-167-3:267"""
"""Pulsed electrical stimulation …",null,"""Orthopedics""","""2006-10-26""","""['Michael AMont', 'David SHung…","""10.3928/01477447-20061001-13"""


In [74]:
with pl.Config(fmt_str_lengths=2000, tbl_rows=2):
    print(test.tail(2))

shape: (2, 6)
┌─────────────────┬─────────────────┬─────────────┬────────────┬─────────────────┬─────────────────┐
│ title           ┆ abstract        ┆ journal     ┆ date       ┆ authors         ┆ doi             │
│ ---             ┆ ---             ┆ ---         ┆ ---        ┆ ---             ┆ ---             │
│ str             ┆ str             ┆ str         ┆ str        ┆ str             ┆ str             │
╞═════════════════╪═════════════════╪═════════════╪════════════╪═════════════════╪═════════════════╡
│ High-throughput ┆ Rapid,          ┆ Methods in  ┆ 2006-10-31 ┆ ['Karen         ┆ 10.1385/1-59745 │
│ quantitation of ┆ quantitative    ┆ molecular   ┆            ┆ JRees-Milton',  ┆ -167-3:267      │
│ metabolically   ┆ methods suited  ┆ biology     ┆            ┆ 'Tassos PAnasta ┆                 │
│ labeled anionic ┆ to a large      ┆ (Clifton,   ┆            ┆ ssiades']       ┆                 │
│ glycoconjugates ┆ number of       ┆ N.J.)       ┆            ┆             

In [22]:
def extract_keyword(text):
    if text is None:
        return []
    r = Rake()
    r.extract_keywords_from_text(str(text))
    return list(dict.fromkeys(r.get_ranked_phrases())) # adding a check to remove duplicates because there seems to be a lot

test_keyword = test.with_columns([
    pl.col('abstract').map_elements(extract_keyword, return_dtype = pl.List(pl.String))
])


In [23]:
test_keyword

title,abstract,journal,date,authors,doi
str,list[str],str,str,str,str
"""Anti-tumor necrosis factor-alp…","[""infliximab treatment significantly lowered"", ""gamma inducible protein"", … ""14""]","""Rheumatology international""","""2006-10-20""","""['YasunoriKageyama', 'EijiTori…","""10.1007/s00296-006-0241-1"""
"""[Quantification of expression …","[""could also remarkably diminish ltb4"", ""lox exciting protein flap inhibitor"", … ""00""]","""Beijing da xue xue bao. Yi xue…","""2006-10-28""","""['Zhan-kunChen', 'Hou-shanLv']""",null
"""Interleukin-7 induced immunopa…","[""7 induces tumour necrosis factor alpha"", ""mig ), macrophage inflammatory protein"", … ""able""]","""Annals of the rheumatic diseas…","""2006-10-14""","""['S A YHartgring', 'J W JBijls…","""10.1136/ard.2006.058479"""
"""High-throughput quantitation o…","[""wallac 1450 microbeta trilux"", ""n )]- glucosamine allows"", … ""3h""]","""Methods in molecular biology (…","""2006-10-31""","""['Karen JRees-Milton', 'Tassos…","""10.1385/1-59745-167-3:267"""
"""Pulsed electrical stimulation …",null,"""Orthopedics""","""2006-10-26""","""['Michael AMont', 'David SHung…","""10.3928/01477447-20061001-13"""


In [24]:
# view length of each abstract
test_keyword.with_columns(
    pl.col('abstract').list.len()
)

title,abstract,journal,date,authors,doi
str,u32,str,str,str,str
"""Anti-tumor necrosis factor-alp…",41,"""Rheumatology international""","""2006-10-20""","""['YasunoriKageyama', 'EijiTori…","""10.1007/s00296-006-0241-1"""
"""[Quantification of expression …",79,"""Beijing da xue xue bao. Yi xue…","""2006-10-28""","""['Zhan-kunChen', 'Hou-shanLv']""",null
"""Interleukin-7 induced immunopa…",72,"""Annals of the rheumatic diseas…","""2006-10-14""","""['S A YHartgring', 'J W JBijls…","""10.1136/ard.2006.058479"""
"""High-throughput quantitation o…",85,"""Methods in molecular biology (…","""2006-10-31""","""['Karen JRees-Milton', 'Tassos…","""10.1385/1-59745-167-3:267"""
"""Pulsed electrical stimulation …",null,"""Orthopedics""","""2006-10-26""","""['Michael AMont', 'David SHung…","""10.3928/01477447-20061001-13"""


In [26]:
# abstract is now a string in a list so we need to add fmt table cell list len...

with pl.Config(fmt_str_lengths=2000, fmt_table_cell_list_len=100, tbl_rows=2):
    print(test_keyword.head(2))

shape: (2, 6)
┌─────────────────┬────────────────┬────────────────┬────────────┬────────────────┬────────────────┐
│ title           ┆ abstract       ┆ journal        ┆ date       ┆ authors        ┆ doi            │
│ ---             ┆ ---            ┆ ---            ┆ ---        ┆ ---            ┆ ---            │
│ str             ┆ list[str]      ┆ str            ┆ str        ┆ str            ┆ str            │
╞═════════════════╪════════════════╪════════════════╪════════════╪════════════════╪════════════════╡
│ Anti-tumor      ┆ ["infliximab   ┆ Rheumatology   ┆ 2006-10-20 ┆ ['YasunoriKage ┆ 10.1007/s00296 │
│ necrosis        ┆ treatment      ┆ international  ┆            ┆ yama',         ┆ -006-0241-1    │
│ factor-alpha    ┆ significantly  ┆                ┆            ┆ 'EijiTorikai', ┆                │
│ antibody        ┆ lowered",      ┆                ┆            ┆ 'AkiraNagano'] ┆                │
│ treatment       ┆ "gamma         ┆                ┆            ┆           

### Save cleaned lazyframe as a csv and then use it to extract keywords

In [60]:
dt_drop_raw_df.sink_csv('pubmed_cleaned_drop_author.csv')

### Extracting keywords for entire dataframe

In [61]:
rcd = pl.scan_csv("/Users/shantibrodnick/Downloads/510Capstone/GenderDisparityinResearch/notebooks/Article_Data_APIs/data/pubmed_cleaned_drop_author.csv")

In [62]:
rcd.select(pl.len()).collect().item()

10206788

In [63]:
def extract_keyword(text):
    if text is None:
        return []
    r = Rake()
    r.extract_keywords_from_text(str(text))
    return list(dict.fromkeys(r.get_ranked_phrases())) 

keyword_rcd = rcd.with_columns([
    pl.col('abstract').map_elements(extract_keyword, return_dtype = pl.List(pl.String))
])

In [64]:
keyword_rcd.select(pl.len()).collect().item()

10206788

In [69]:
keyword_rcd_test = keyword_rcd.head(15).collect()

In [71]:
keyword_rcd_test

title,abstract,journal,date,doi
str,list[str],str,str,str
"""Anti-tumor necrosis factor-alp…","[""infliximab treatment significantly lowered"", ""gamma inducible protein"", … ""14""]","""Rheumatology international""","""2006-10-20""","""10.1007/s00296-006-0241-1"""
"""[Quantification of expression …","[""could also remarkably diminish ltb4"", ""lox exciting protein flap inhibitor"", … ""00""]","""Beijing da xue xue bao. Yi xue…","""2006-10-28""",null
"""Interleukin-7 induced immunopa…","[""7 induces tumour necrosis factor alpha"", ""mig ), macrophage inflammatory protein"", … ""able""]","""Annals of the rheumatic diseas…","""2006-10-14""","""10.1136/ard.2006.058479"""
"""High-throughput quantitation o…","[""wallac 1450 microbeta trilux"", ""n )]- glucosamine allows"", … ""3h""]","""Methods in molecular biology (…","""2006-10-31""","""10.1385/1-59745-167-3:267"""
"""Pulsed electrical stimulation …",null,"""Orthopedics""","""2006-10-26""","""10.3928/01477447-20061001-13"""
…,…,…,…,…
"""Heat shock proteins induce T c…","[""antigen specific immunotherapy approach involving modulation"", ""bacterial hsps inhibited disease development"", … ""anti""]","""Annals of the rheumatic diseas…","""2006-10-14""","""10.1136/ard.2006.058495"""
"""Detection of novel intracellul…","[""synuclein oligomerization using fluorescence lifetime imaging"", ""carboxy terminus interaction within single alpha"", … ""adopt""]","""FASEB journal : official publi…","""2006-10-03""","""10.1096/fj.05-5422com"""
"""AMG 531, a thrombopoiesis-stim…","[""amg 531 per kilogram per week"", ""receive six weekly subcutaneous injections"", … ""0""]","""The New England journal of med…","""2006-10-20""","""10.1056/NEJMoa054626"""


In [70]:
keyword_rcd_test.with_columns(
    pl.col('abstract').list.len()
)

title,abstract,journal,date,doi
str,u32,str,str,str
"""Anti-tumor necrosis factor-alp…",41,"""Rheumatology international""","""2006-10-20""","""10.1007/s00296-006-0241-1"""
"""[Quantification of expression …",79,"""Beijing da xue xue bao. Yi xue…","""2006-10-28""",null
"""Interleukin-7 induced immunopa…",72,"""Annals of the rheumatic diseas…","""2006-10-14""","""10.1136/ard.2006.058479"""
"""High-throughput quantitation o…",85,"""Methods in molecular biology (…","""2006-10-31""","""10.1385/1-59745-167-3:267"""
"""Pulsed electrical stimulation …",null,"""Orthopedics""","""2006-10-26""","""10.3928/01477447-20061001-13"""
…,…,…,…,…
"""Heat shock proteins induce T c…",73,"""Annals of the rheumatic diseas…","""2006-10-14""","""10.1136/ard.2006.058495"""
"""Detection of novel intracellul…",48,"""FASEB journal : official publi…","""2006-10-03""","""10.1096/fj.05-5422com"""
"""AMG 531, a thrombopoiesis-stim…",96,"""The New England journal of med…","""2006-10-20""","""10.1056/NEJMoa054626"""


In [ ]:
# Check estimated size
keyword_rcd.collect().estimated_size("mb")
# 10.2 GB

10279.678154945374

# Try and pull gender or something like that out of the full abstract